# Übung 2.0 — Simulation einer Vorlesungssaal-Lüftung

In dieser Übung simulieren Sie das Strömungsszenario aus der Aufgabenstellung
mit dem beigelegten Finite-Volumen-Löser `solver.py`.

Die Fallkonfiguration wird als Dictionary definiert.
Untenstehend finden Sie ein **Beispiel** -- passen Sie dieses
mit Ihren Ergebnissen aus Aufgabe a) an.

## Fallkonfiguration

Die Funktion `init(grid, state)` definiert die Geometrie (Objekte) und Quellterme auf dem Gitter.
Der Löser ruft sie einmal vor der Zeitschleife auf.

### Objekte

Objekte werden als binäre Masken auf dem Gitter definiert. Jedes Objekt hat folgende Eigenschaften:

| Eigenschaft       | Beschreibung |
|-------------------|--------------|
| `Obj[i, :, :]`    | Maske: 1 wo das Objekt existiert |
| `Obj_adiabatic[i]`| 1 = adiabatisch (kein Wärmefluss durch die Oberfläche) |
| `Obj_overwrite[i]` | 1 = Feldwerte werden jeden Zeitschritt überschrieben (z.B. Einlass) |
| `Obj_T[i]`         | Temperaturwert im Objekt (Dirichlet) |
| `Obj_Y[i]`         | CO₂-Konzentration im Objekt |
| `Obj_U1[i]`, `Obj_U2[i]` | Geschwindigkeitskomponenten im Objekt |

### Quellterme

Volumenquellterme `Source_T` und `Source_Y` werden als 2D-Arrays auf dem Gitter definiert.
Der Löser addiert sie in jeder Zelle pro Zeitschritt.

### Gitterkonvention

- `x1` = horizontale Richtung, `x2` = vertikale Richtung (Auftriebs-Richtung)
- Indexierung: `[ix2, ix1]` (Zeile = Höhe, Spalte = Breite)
- Ghostzellen: Index `0` und `-1` sind Randzellen

### Dimensionen
Alle folgenden Werte sind dimensionslos.

## Implementation des Lösers 

Muss einmal ausgeführt werden, damit solver_run unten verfügbar ist. Interessierte können hier das Innere eines CFD-Lösers erkunden.

In [ ]:
"""
Simple 2D Finite Volume CFD Solver
Based on Patrick Jenny's FVM code for ETH MAS Fire Safety Engineering.
Refactored to accept case configurations as dictionaries.
"""

import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython import display as dsp
except ImportError:
    dsp = None


def make_grid(lx1, lx2, nx1, nx2):
    """Build uniform Cartesian grid with ghost cells."""
    dx1 = lx1 / nx1
    dx2 = lx2 / nx2
    x1 = dx1 * (np.arange(nx1 + 2) - 0.5)
    x2 = dx2 * (np.arange(nx2 + 2) - 0.5)
    X1, X2 = np.meshgrid(x1, x2)
    return dict(lx1=lx1, lx2=lx2, nx1=nx1, nx2=nx2,
                dx1=dx1, dx2=dx2, x1=x1, x2=x2, X1=X1, X2=X2)


def make_state(grid, no_max=10):
    """Allocate all field arrays (zero-initialized)."""
    nx1, nx2 = grid["nx1"], grid["nx2"]
    shape = (nx2 + 2, nx1 + 2)
    obj_shape = (no_max, nx2 + 2, nx1 + 2)

    return dict(
        # Objects
        no=0, no_max=no_max,
        Obj=np.zeros(obj_shape),
        Obj_adiabatic=np.zeros(no_max),
        Obj_overwrite=np.zeros(no_max),
        Obj_T=np.zeros(no_max),
        Obj_Y=np.zeros(no_max),
        Obj_U1=np.zeros(no_max),
        Obj_U2=np.zeros(no_max),
        # Source terms
        Source_U1=np.zeros(shape),
        Source_U2=np.zeros(shape),
        Source_T=np.zeros(shape),
        Source_Y=np.zeros(shape),
        # Solution fields
        U1=np.zeros(shape), U2=np.zeros(shape),
        P=np.zeros(shape),  T=np.zeros(shape),
        Y=np.zeros(shape),
    )


def _minmod_slope(phi, dx, axis):
    """Compute limited slope for second-order upwinding."""
    if axis == 1:  # x1 direction
        sp = np.zeros_like(phi)
        sm = np.zeros_like(phi)
        sp[1:-1, 1:-1] = (phi[1:-1, 2:] - phi[1:-1, 1:-1]) / dx
        sm[1:-1, 1:-1] = (phi[1:-1, 1:-1] - phi[1:-1, 0:-2]) / dx
    else:  # x2 direction
        sp = np.zeros_like(phi)
        sm = np.zeros_like(phi)
        sp[1:-1, 1:-1] = (phi[2:, 1:-1] - phi[1:-1, 1:-1]) / dx
        sm[1:-1, 1:-1] = (phi[1:-1, 1:-1] - phi[0:-2, 1:-1]) / dx
    return (sp * sm > 0) * (
        (sp > 0) * ((sp > sm) * sm + (sp < sm) * sp) +
        (sp < 0) * ((sp > sm) * sp + (sp < sm) * sm)
    )


def solver_run(case):
    """
    Run the FVM solver for a given case configuration.

    Parameters
    ----------
    case : dict with keys
        Grid:     lx1, lx2, nx1, nx2
        Physics:  Re, Pr, Sc, Fr, T_max
        Solver:   t_end, nJacobi, nPlot, cfl_diff, cfl, bool_up, bool_2nd
        Setup:    init(grid, state) — modifies state in-place
    """
    # --- Grid ---
    grid = make_grid(case["lx1"], case["lx2"], case["nx1"], case["nx2"])
    nx1, nx2 = grid["nx1"], grid["nx2"]
    dx1, dx2 = grid["dx1"], grid["dx2"]
    X1, X2 = grid["X1"], grid["X2"]
    shape = (nx2 + 2, nx1 + 2)

    # --- Physics ---
    Re = case["Re"]
    Pr = case["Pr"]
    Sc = case["Sc"]
    Fr = case["Fr"]
    T_max = case.get("T_max", 1e6)

    # --- Solver params ---
    t_end    = case.get("t_end", 200)
    nt       = case.get("nt", 2_000_000)
    nJacobi  = case.get("nJacobi", 15)
    nPlot    = case.get("nPlot", 10)
    cfl_diff = case.get("cfl_diff", 0.5)
    cfl      = case.get("cfl", 0.5)
    bool_up  = case.get("bool_up", True)
    bool_2nd = case.get("bool_2nd", True)

    # --- State ---
    state = make_state(grid)
    case["init"](grid, state)

    no = state["no"]
    Obj           = state["Obj"]
    Obj_adiabatic = state["Obj_adiabatic"]
    Obj_overwrite = state["Obj_overwrite"]
    Obj_T  = state["Obj_T"]
    Obj_Y  = state["Obj_Y"]
    Obj_U1 = state["Obj_U1"]
    Obj_U2 = state["Obj_U2"]
    Source_U1 = state["Source_U1"]
    Source_U2 = state["Source_U2"]
    Source_T  = state["Source_T"]
    Source_Y  = state["Source_Y"]
    U1 = state["U1"]
    U2 = state["U2"]
    P  = state["P"]
    T  = state["T"]
    Y  = state["Y"]

    # --- Intermediate fields ---
    Uf1  = np.zeros(shape); Uf2  = np.zeros(shape)
    sm   = np.zeros(shape); sp   = np.zeros(shape)
    s1   = np.zeros(shape); s2   = np.zeros(shape)
    H1   = np.zeros(shape); H2   = np.zeros(shape)
    divU = np.zeros(shape); divH = np.zeros(shape)
    tmp  = np.zeros(shape)
    time = 0.0

    # --- Masks ---
    dt_diff = cfl_diff * min(Re, Re * Pr) / (2 / dx1**2 + 2 / dx2**2)

    not_solid           = np.ones(shape)
    not_solid_adiabatic = np.ones(shape)
    not_overwrite       = np.ones(shape)
    for io in range(no):
        if Obj_overwrite[io] == 1.0:
            not_overwrite = not_overwrite * (1.0 - Obj[io])
        if Obj_overwrite[io] == 0.0:
            not_solid = not_solid * (1.0 - Obj[io])
        if Obj_adiabatic[io] == 1.0:
            not_solid_adiabatic = not_solid_adiabatic * (1.0 - Obj[io])

    # --- Plot setup ---
    fig = plt.figure(figsize=(20, 10))
    sf1 = fig.add_subplot(221)
    sf2 = fig.add_subplot(222)
    sf3 = fig.add_subplot(223)
    sf4 = fig.add_subplot(224)
    for ax in [sf1, sf2, sf3, sf4]:
        ax.axis("scaled")

    def my_plot():
        speed = (U1[1:-1, 1:-1]**2 + U2[1:-1, 1:-1]**2)**0.5
        smax = speed.max()
        if smax == 0:
            smax = 1
        speed = speed / smax

        Y_plot = np.minimum(Y, 1)

        sf1.clear(); sf2.clear(); sf3.clear(); sf4.clear()
        sf3.contourf(X1[1:-1, 1:-1], X2[1:-1, 1:-1],
                     (U1[1:-1, 1:-1]**2 + U2[1:-1, 1:-1]**2)**0.5,
                     200, cmap="Blues")
        sf2.contourf(X1[1:-1, 1:-1], X2[1:-1, 1:-1],
                     T[1:-1, 1:-1], 200, cmap="magma")
        sf1.contourf(X1[1:-1, 1:-1], X2[1:-1, 1:-1],
                     not_solid[1:-1, 1:-1] + 0.001, [0.5, 1.5],
                     colors="lightgrey")
        sf1.contourf(X1[1:-1, 1:-1], X2[1:-1, 1:-1],
                     Y_plot[1:-1, 1:-1], 200, cmap="magma")
        sf4.streamplot(X1[1:-1, 1:-1], X2[1:-1, 1:-1],
                       U1[1:-1, 1:-1], U2[1:-1, 1:-1],
                       color=speed, cmap="Blues", density=2)
        sf1.set_title(f"CO₂")
        sf2.set_title("Temperature")
        sf3.set_title("|U|")
        sf4.set_title("Streamlines")

       
        fig.tight_layout()
        for ax in [sf1, sf2, sf3, sf4]:
            ax.axis("scaled")
        if dsp is not None:
            dsp.clear_output(wait=True)
            dsp.display(fig)
        else:
            plt.pause(0.01)

        print(f"t = {time:.2f}")
        print(f"Max speed: {smax:.2f}")


    # ===================== TIME LOOP =====================
    for it in range(nt + 1):
        if time >= t_end:
            break

        # --- Wall BCs (adiabatic, no-slip) ---
        # bottom / top
        U1[ 0, 1:-1] = -U1[ 1, 1:-1]
        U1[-1, 1:-1] = -U1[-2, 1:-1]
        U2[ 0, 1:-1] = -U2[ 1, 1:-1]
        U2[-1, 1:-1] = -U2[-2, 1:-1]
        T [ 0, 1:-1] =  T [ 1, 1:-1]
        T [-1, 1:-1] =  T [-2, 1:-1]
        Y [ 0, 1:-1] =  Y [ 1, 1:-1]
        Y [-1, 1:-1] =  Y [-2, 1:-1]
        # left / right
        U1[1:-1,  0] = -U1[1:-1,  1]
        U1[1:-1, -1] = -U1[1:-1, -2]
        U2[1:-1,  0] = -U2[1:-1,  1]
        U2[1:-1, -1] = -U2[1:-1, -2]
        T [1:-1,  0] =  T [1:-1,  1]
        T [1:-1, -1] =  T [1:-1, -2]
        Y [1:-1,  0] =  Y [1:-1,  1]
        Y [1:-1, -1] =  Y [1:-1, -2]

        # --- Object BCs ---
        Y  = Y  * not_solid * not_overwrite
        T  = T  * not_solid * not_overwrite
        U1 = U1 * not_solid * not_overwrite
        U2 = U2 * not_solid * not_overwrite
        for io in range(no):
            T  = T  + Obj_T [io] * Obj[io]
            Y  = Y  + Obj_Y [io] * Obj[io]
            U1 = U1 + Obj_U1[io] * Obj[io]
            U2 = U2 + Obj_U2[io] * Obj[io]

        # --- Advection velocity ---
        Uf2[1:-1, 1:-1] = (U2[0:-2, 1:-1] + U2[1:-1, 1:-1]) / 2
        Uf1[1:-1, 1:-1] = (U1[1:-1, 0:-2] + U1[1:-1, 1:-1]) / 2

        # --- Time step ---
        if bool_up:
            dt = min(dt_diff, cfl / ((abs(U1) / dx1 + abs(U2) / dx2).max() + 1e-30))
        else:
            dt = min(dt_diff, 1 / dt_diff / ((abs(U1) / dx1 + abs(U2) / dx2).max() + 1e-30)**2)

        # ---- Slopes for 2nd-order upwind ----
        def _slopes(phi):
            s1_ = _minmod_slope(phi, dx1, axis=1)
            s2_ = _minmod_slope(phi, dx2, axis=0)
            return s1_, s2_

        def _upwind_2nd(phi, s1_, s2_):
            return (
                not_solid[0:-2, 1:-1] * Uf2[1:-1, 1:-1] * ((Uf2[1:-1, 1:-1]>0)*(phi[0:-2, 1:-1] + s2_[0:-2, 1:-1]*(dx2/2 - Uf2[1:-1, 1:-1]*dt/2)) + (Uf2[1:-1, 1:-1]<0)*(phi[1:-1, 1:-1] + s2_[1:-1, 1:-1]*(-dx2/2 - Uf2[1:-1, 1:-1]*dt/2))) / dx2 -
                not_solid[2:  , 1:-1] * Uf2[2:  , 1:-1] * ((Uf2[2:  , 1:-1]>0)*(phi[1:-1, 1:-1] + s2_[1:-1, 1:-1]*(dx2/2 - Uf2[2:  , 1:-1]*dt/2)) + (Uf2[2:  , 1:-1]<0)*(phi[2:  , 1:-1] + s2_[2:  , 1:-1]*(-dx2/2 - Uf2[2:  , 1:-1]*dt/2))) / dx2 +
                not_solid[1:-1, 0:-2] * Uf1[1:-1, 1:-1] * ((Uf1[1:-1, 1:-1]>0)*(phi[1:-1, 0:-2] + s1_[1:-1, 0:-2]*(dx1/2 - Uf1[1:-1, 1:-1]*dt/2)) + (Uf1[1:-1, 1:-1]<0)*(phi[1:-1, 1:-1] + s1_[1:-1, 1:-1]*(-dx1/2 - Uf1[1:-1, 1:-1]*dt/2))) / dx1 -
                not_solid[1:-1, 2:  ] * Uf1[1:-1, 2:  ] * ((Uf1[1:-1, 2:  ]>0)*(phi[1:-1, 1:-1] + s1_[1:-1, 1:-1]*(dx1/2 - Uf1[1:-1, 2:  ]*dt/2)) + (Uf1[1:-1, 2:  ]<0)*(phi[1:-1, 2:  ] + s1_[1:-1, 2:  ]*(-dx1/2 - Uf1[1:-1, 2:  ]*dt/2))) / dx1
            )

        def _upwind_1st(phi):
            return (
                not_solid[0:-2, 1:-1] * Uf2[1:-1, 1:-1] * ((Uf2[1:-1, 1:-1]>0)*phi[0:-2, 1:-1] + (Uf2[1:-1, 1:-1]<0)*phi[1:-1, 1:-1]) / dx2 -
                not_solid[2:  , 1:-1] * Uf2[2:  , 1:-1] * ((Uf2[2:  , 1:-1]>0)*phi[1:-1, 1:-1] + (Uf2[2:  , 1:-1]<0)*phi[2:  , 1:-1]) / dx2 +
                not_solid[1:-1, 0:-2] * Uf1[1:-1, 1:-1] * ((Uf1[1:-1, 1:-1]>0)*phi[1:-1, 0:-2] + (Uf1[1:-1, 1:-1]<0)*phi[1:-1, 1:-1]) / dx1 -
                not_solid[1:-1, 2:  ] * Uf1[1:-1, 2:  ] * ((Uf1[1:-1, 2:  ]>0)*phi[1:-1, 1:-1] + (Uf1[1:-1, 2:  ]<0)*phi[1:-1, 2:  ]) / dx1
            )

        def _central(phi, vel1, vel2):
            return (
                not_solid[0:-2, 1:-1] * (phi[0:-2, 1:-1]*vel2[0:-2, 1:-1] + phi[1:-1, 1:-1]*vel2[1:-1, 1:-1]) / 2 / dx2 -
                not_solid[2:  , 1:-1] * (phi[2:  , 1:-1]*vel2[2:  , 1:-1] + phi[1:-1, 1:-1]*vel2[1:-1, 1:-1]) / 2 / dx2 +
                not_solid[1:-1, 0:-2] * (phi[1:-1, 0:-2]*vel1[1:-1, 0:-2] + phi[1:-1, 1:-1]*vel1[1:-1, 1:-1]) / 2 / dx1 -
                not_solid[1:-1, 2:  ] * (phi[1:-1, 2:  ]*vel1[1:-1, 2:  ] + phi[1:-1, 1:-1]*vel1[1:-1, 1:-1]) / 2 / dx1
            )

        def _convection(phi, vel1=None, vel2=None):
            """Compute convective flux with selected scheme."""
            if bool_up and bool_2nd:
                s1_, s2_ = _slopes(phi)
                return _upwind_2nd(phi, s1_, s2_)
            elif bool_up:
                return _upwind_1st(phi)
            else:
                if vel1 is None: vel1 = U1
                if vel2 is None: vel2 = U2
                return _central(phi, vel1, vel2)

        # --- Diffusion helper ---
        def _laplacian(phi, ns):
            return (
                ns[0:-2, 1:-1] * (phi[0:-2, 1:-1] - phi[1:-1, 1:-1]) / dx2**2 +
                ns[2:  , 1:-1] * (phi[2:  , 1:-1] - phi[1:-1, 1:-1]) / dx2**2 +
                ns[1:-1, 0:-2] * (phi[1:-1, 0:-2] - phi[1:-1, 1:-1]) / dx1**2 +
                ns[1:-1, 2:  ] * (phi[1:-1, 2:  ] - phi[1:-1, 1:-1]) / dx1**2
            )

        # ---- H1 (x1-momentum RHS) ----
        H1[1:-1, 1:-1] = (
            (U1[0:-2, 1:-1] - 2*U1[1:-1, 1:-1] + U1[2:, 1:-1]) / dx2**2 +
            (U1[1:-1, 0:-2] - 2*U1[1:-1, 1:-1] + U1[1:-1, 2:]) / dx1**2
        ) / Re + Source_U1[1:-1, 1:-1]
        if bool_up and bool_2nd:
            s1U1, s2U1 = _slopes(U1)
            H1[1:-1, 1:-1] += _upwind_2nd(U1, s1U1, s2U1)
        elif bool_up:
            H1[1:-1, 1:-1] += _upwind_1st(U1)
        else:
            H1[1:-1, 1:-1] += _central(U1, U1, U2)

        # ---- H2 (x2-momentum RHS) ----
        H2[1:-1, 1:-1] = (
            (U2[0:-2, 1:-1] - 2*U2[1:-1, 1:-1] + U2[2:, 1:-1]) / dx2**2 +
            (U2[1:-1, 0:-2] - 2*U2[1:-1, 1:-1] + U2[1:-1, 2:]) / dx1**2
        ) / Re + Source_U2[1:-1, 1:-1]
        if bool_up and bool_2nd:
            s1U2, s2U2 = _slopes(U2)
            H2[1:-1, 1:-1] += _upwind_2nd(U2, s1U2, s2U2)
        elif bool_up:
            H2[1:-1, 1:-1] += _upwind_1st(U2)
        else:
            H2[1:-1, 1:-1] += _central(U2, U1, U2)

        # Buoyancy
        H2[1:-1, 1:-1] += T[1:-1, 1:-1] / Fr**2

        # H BCs
        for H in [H1, H2]:
            H[ 0, 1:-1] = H[ 1, 1:-1]; H[-1, 1:-1] = H[-2, 1:-1]
            H[1:-1,  0] = H[1:-1,  1]; H[1:-1, -1] = H[1:-1, -2]

        # ---- Divergences ----
        divH[1:-1, 1:-1] = (
            not_solid[2:, 1:-1]  * (H2[2:, 1:-1]  + H2[1:-1, 1:-1]) / 2 / dx2 -
            not_solid[0:-2, 1:-1]* (H2[0:-2, 1:-1] + H2[1:-1, 1:-1]) / 2 / dx2 +
            not_solid[1:-1, 2:]  * (H1[1:-1, 2:]  + H1[1:-1, 1:-1]) / 2 / dx1 -
            not_solid[1:-1, 0:-2]* (H1[1:-1, 0:-2] + H1[1:-1, 1:-1]) / 2 / dx1
        )
        divU[1:-1, 1:-1] = (
            not_solid[2:, 1:-1]  * (U2[2:, 1:-1]  + U2[1:-1, 1:-1]) / 2 / dx2 -
            not_solid[0:-2, 1:-1]* (U2[0:-2, 1:-1] + U2[1:-1, 1:-1]) / 2 / dx2 +
            not_solid[1:-1, 2:]  * (U1[1:-1, 2:]  + U1[1:-1, 1:-1]) / 2 / dx1 -
            not_solid[1:-1, 0:-2]* (U1[1:-1, 0:-2] + U1[1:-1, 1:-1]) / 2 / dx1
        )

        # ---- Pressure Poisson (Jacobi) ----
        for k in range(nJacobi):
            P[ 0, 1:-1] = P[ 1, 1:-1]
            P[-1, 1:-1] = P[-2, 1:-1]
            P[1:-1,  0] = P[1:-1,  1]
            P[1:-1, -1] = P[1:-1, -2]
            P[ 0, 1:-1] -= T[ 1, 1:-1] / Fr**2 * dx2
            P[-1, 1:-1] += T[-2, 1:-1] / Fr**2 * dx2

            tmp[1:-1, 1:-1] = (
                (not_solid[2:, 1:-1]*P[2:, 1:-1] + not_solid[0:-2, 1:-1]*P[0:-2, 1:-1]) / dx2**2 +
                (not_solid[1:-1, 2:]*P[1:-1, 2:] + not_solid[1:-1, 0:-2]*P[1:-1, 0:-2]) / dx1**2 -
                divH[1:-1, 1:-1] - divU[1:-1, 1:-1] / dt
            )
            tmp[1:-1, 1:-1] += (
                (not_solid[0:-2, 1:-1] - 1) * T[1:-1, 1:-1] -
                (not_solid[2:, 1:-1]   - 1) * T[1:-1, 1:-1]
            ) / Fr**2 / dx2
            P[1:-1, 1:-1] = 0.5 * P[1:-1, 1:-1] + 0.5 * tmp[1:-1, 1:-1] / (
                (not_solid[1:-1, 0:-2] + not_solid[1:-1, 2:]) / dx1**2 +
                (not_solid[0:-2, 1:-1] + not_solid[2:, 1:-1]) / dx2**2 +
                (not_solid[1:-1, 1:-1] - 1.0)
            )

        # ---- Velocity update ----
        U1[1:-1, 1:-1] += (
            H1[1:-1, 1:-1] + (
                not_solid[1:-1, 0:-2]*P[1:-1, 0:-2] + (1 - not_solid[1:-1, 0:-2])*P[1:-1, 1:-1] -
                not_solid[1:-1, 2:]  *P[1:-1, 2:]   - (1 - not_solid[1:-1, 2:])  *P[1:-1, 1:-1]
            ) / 2 / dx1
        ) * dt
        U2[1:-1, 1:-1] += (
            H2[1:-1, 1:-1] + (
                not_solid[0:-2, 1:-1]*P[0:-2, 1:-1] + (1 - not_solid[0:-2, 1:-1])*(P[1:-1, 1:-1] - T[1:-1, 1:-1]/Fr**2*dx2) -
                not_solid[2:, 1:-1]  *P[2:, 1:-1]   - (1 - not_solid[2:, 1:-1])  *(P[1:-1, 1:-1] + T[1:-1, 1:-1]/Fr**2*dx2)
            ) / 2 / dx2
        ) * dt

        # Object BCs for velocity
        P  = P  * not_solid
        U1 = U1 * not_solid * not_overwrite
        U2 = U2 * not_solid * not_overwrite
        for io in range(no):
            U1 = U1 + Obj_U1[io] * Obj[io]
            U2 = U2 + Obj_U2[io] * Obj[io]

        # ---- Temperature update ----
        tmp[1:-1, 1:-1] = _laplacian(T, not_solid_adiabatic) / Re / Pr + Source_T[1:-1, 1:-1]
        tmp[1:-1, 1:-1] += _convection(T, U1, U2)
        T[1:-1, 1:-1] += tmp[1:-1, 1:-1] * dt

        # ---- Scalar (CO2) update ----
        tmp[1:-1, 1:-1] = _laplacian(Y, not_solid) / Re / Sc + Source_Y[1:-1, 1:-1]
        tmp[1:-1, 1:-1] += _convection(Y, U1, U2)
        Y[1:-1, 1:-1] += tmp[1:-1, 1:-1] * dt

        # Object BCs for T, Y
        Y = Y * not_solid * not_overwrite
        T = T * not_solid * not_overwrite
        T = T * (T < T_max) + T_max * (T >= T_max)
        for io in range(no):
            T = T + Obj_T[io] * Obj[io]
            Y = Y + Obj_Y[io] * Obj[io]

        # ---- Plot ----
        if it % nPlot == 0 or time >= t_end:
            my_plot()

        time += min(dt, t_end - time)

    my_plot()
    return dict(grid=grid, U1=U1, U2=U2, P=P, T=T, Y=Y, time=time)

## Beispielfälle

In [ ]:

def init_example_1(grid, state):
    nx1, nx2 = grid["nx1"], grid["nx2"]
    dx1, dx2 = grid["dx1"], grid["dx2"]
    lx1, lx2 = grid["lx1"], grid["lx2"]
    
    # -- Dimensionslose Quellen --
    Q_star_total = 0.1 # Abwärme
    S_star_total = 0.1 # Atemflussrate * Konzentration
    
    # -- Dimensionslose Einlassbedingungen --
    U_in_star = 0.5 # Einlassgeschwindigkeit
    T_in_star = 0.0 # Einlasstemperatur
    Y_in_star = 0.0 # Einlasskonzentration
    
    # ====== Umwandlung in Gitterindizes ======
    def ix1_of(x): return int(round(x / dx1))
    def ix2_of(x): return int(round(x / dx2))

    # Eck-Indizes des soliden Objekts, der Hitzequelle und der CO2-Quelle

    solid_left = ix1_of(0.25)
    solid_right = ix1_of(0.75)
    solid_bottom = ix2_of(0.5 - 0.05)
    solid_top = ix2_of(0.5 + 0.05)
    
    heatcube_left = ix1_of(0.25 - 0.1)
    heatcube_right = ix1_of(0.25 + 0.1)
    heatcube_bottom = ix2_of(0.5 - 0.1)
    heatcube_top = ix2_of(0.5 + 0.1)

    coldcube_left = ix1_of(0.75 - 0.1)
    coldcube_right = ix1_of(0.75 + 0.1)
    coldcube_bottom = ix2_of(0.5 - 0.1)
    coldcube_top = ix2_of(0.5 + 0.1)

    co2_source_left = ix1_of(0.5 - 0.1)
    co2_source_right = ix1_of(0.5 + 0.1)
    co2_source_bottom = ix2_of(0.25 - 0.1)
    co2_source_top = ix2_of(0.25 + 0.1)



    

    i_inlet  = ix2_of(0.1)
    i_outlet = ix2_of(lx2 - 0.1)
    
    # ====== Objekte ======
    state["no"] = 5
    Obj = state["Obj"]
    
    # Objekt 0: Festkörper
    state["Obj_adiabatic"][0] = 1.0
    Obj[0,  solid_bottom:solid_top, solid_left:solid_right] = 1

    # Objekt 1: Kalter Wuerfel
    state["Obj_T"][1] = -1.0
    Obj[1,  coldcube_bottom:coldcube_top, coldcube_left:coldcube_right] = 1

    # Objekt 2: Heisser Wuerfel
    state["Obj_T"][2] = 1.0
    Obj[2,  heatcube_bottom:heatcube_top, heatcube_left:heatcube_right] = 1

    return 
    # CO2-Quelle
    co2_mask = np.zeros((nx2+2, nx1+2))
    co2_mask[co2_source_bottom:co2_source_top, co2_source_left:co2_source_right] = 0.01
    n_mouth = co2_mask.sum()

    state["Source_Y"][:] = co2_mask * 1.0

    # Objekt 3: Einlass
    state["Obj_overwrite"][3] = 1.0
    state["Obj_U1"][3] = U_in_star     # horizontale Einströmgeschwindigkeit
    state["Obj_T"][3]  = T_in_star
    state["Obj_Y"][3]  = Y_in_star
    Obj[3, 0:i_inlet+1, [0,1]] = 1.0        # linke Wand, unterer Teil
    
    # Objekt 4: Auslass
    state["Obj_overwrite"][4] = 1.0
    Obj[4, 0:i_inlet+1, [-2,-1]] = 1.0    # rechte Wand, oberer Teil
    state["Obj_U1"][4] = U_in_star
    
    return 
    # ====== Quellterme ======

    # Wärmequelle

    heat_mask = np.zeros((nx2+2, nx1+2))
    heat_mask[heatcube_bottom:heatcube_top, heatcube_left:heatcube_right] = 1.0
    heat_mask[solid_bottom:solid_top, solid_left:solid_right] = 0.0  # Wichtig: Solides Objekt aussparen, sonst geht Wärme im Objekt verloren.

    n_shell = heat_mask.sum()
    
    if n_shell > 0:
        # Wärmeflussdichte: Totale Wärme / Fläche (in 2D)
        state["Source_T"][:] = heat_mask * Q_star_total / (n_shell * dx1 * dx2)
    
    

example_case_1 = dict(
    lx1 = 1.0,     # L_x / L∞
    lx2 = 1.0,     # L_y / L∞
    nx1 = 100,
    nx2 = 100,
    
    # Dimensionslose Kennzahlen 
    Re = 100,
    Pr = 1.7,
    Sc = 10000.0, 
    Fr = 0.1,
    
    T_max = 1.0,
    
    # Weitere Parameter
    t_end   = 200,
    nJacobi = 15,
    nPlot   = 50,

    # Oben definierte Initialisierungsfunktion
    init=init_example_1
)



In [ ]:

def init_example_2(grid, state):
    nx1, nx2 = grid["nx1"], grid["nx2"]
    dx1, dx2 = grid["dx1"], grid["dx2"]
    lx1, lx2 = grid["lx1"], grid["lx2"]
    
    
    # -- Dimensionslose Quellen --
    Q_star_total = 0.1 # Abwärme
    S_star_total = 0.1 # Atemflussrate * Konzentration
    
    # -- Dimensionslose Einlassbedingungen --
    U_in_star = 0.5 # Einlassgeschwindigkeit
    T_in_star = 0.0 # Einlasstemperatur
    Y_in_star = 0.0 # Einlasskonzentration
    
    # ====== Umwandlung in Gitterindizes ======
    def ix1_of(x): return int(round(x / dx1))
    def ix2_of(x): return int(round(x / dx2))

    # Eck-Indizes des soliden Objekts, der Hitzequelle und der CO2-Quelle

    solid_left = ix1_of(0.25)
    solid_right = ix1_of(0.75)
    solid_bottom = ix2_of(0.5 - 0.05)
    solid_top = ix2_of(0.5 + 0.05)
    
    heatcube_left = ix1_of(0.25 - 0.1)
    heatcube_right = ix1_of(0.25 + 0.1)
    heatcube_bottom = ix2_of(0.5 - 0.1)
    heatcube_top = ix2_of(0.5 + 0.1)

    co2cube_left = ix1_of(0.75 - 0.1)
    co2cube_right = ix1_of(0.75 + 0.1)
    co2cube_bottom = ix2_of(0.5 - 0.1)
    co2cube_top = ix2_of(0.5 + 0.1)

    i_inlet  = ix2_of(0.1)
    i_outlet = ix2_of(lx2 - 0.1)
    
    # ====== Objekte ======
    state["no"] = 3
    Obj = state["Obj"]
    
    # Objekt 0: Festkörper
    state["Obj_adiabatic"][0] = 1.0
    state["Obj_overwrite"][0] = 1.0
    Obj[0,  solid_bottom:solid_top, solid_left:solid_right] = 1
    
    # Objekt 1: Einlass
    state["Obj_overwrite"][1] = 1.0
    state["Obj_U1"][1] = U_in_star     # horizontale Einströmgeschwindigkeit
    state["Obj_T"][1]  = T_in_star
    state["Obj_Y"][1]  = Y_in_star
    Obj[1, 0:i_inlet+1, [0,1]] = 1.0        # linke Wand, unterer Teil
    
    # Objekt 2: Auslass
    state["Obj_overwrite"][2] = 1.0
    
    Obj[2, i_outlet:-1, [-2,-1]] = 1.0    # rechte Wand, oberer Teil

    state["Obj_U1"][2] = U_in_star
    
    # ====== Quellterme ======

    # Wärmequelle

    heat_mask = np.zeros((nx2+2, nx1+2))
    heat_mask[heatcube_bottom:heatcube_top, heatcube_left:heatcube_right] = 1.0
    heat_mask[solid_bottom:solid_top, solid_left:solid_right] = 0.0  # Wichtig: Solides Objekt aussparen, sonst geht Wärme im Objekt verloren.

    n_shell = heat_mask.sum()
    
    if n_shell > 0:
        # Wärmeflussdichte: Totale Wärme / Fläche (in 2D)
        state["Source_T"][:] = heat_mask * Q_star_total / (n_shell * dx1 * dx2)
    
    # CO2-Quelle
    co2_mask = np.zeros((nx2+2, nx1+2))
    co2_mask[co2cube_bottom:co2cube_top, co2cube_left:co2cube_right] = 1.0
    n_mouth = co2_mask.sum()

    if n_mouth > 0:
        # CO2-Flussdichte: Totale CO2-Rate / Fläche
        state["Source_Y"][:] = co2_mask * S_star_total / (n_mouth * dx1 * dx2)

example_case_2 = dict(
    lx1 = 1.0,     # L_x / L∞
    lx2 = 1.0,     # L_y / L∞
    nx1 = 64,
    nx2 = 64,
    
    # Dimensionslose Kennzahlen 
    Re = 10000,
    Pr = 1.0,
    Sc = 1.0, 
    Fr = 0.1,
    
    T_max = 1.0,
    
    # Weitere Parameter
    t_end   = 200,
    nJacobi = 15,
    nPlot   = 50,

    # Oben definierte Initialisierungsfunktion
    init=init_example_2
)



## Vorlesungssaal-Szenario [Aufgabe b)]

In [ ]:

def init_exercise(grid, state):
    # TODO: Implementieren sie das Szenario vom Aufgabenblatt
    # Tipp: Alle dafür benötigten Funktionalitäten des Lösers werden im Beispiel oben benutzt.
       
    nx1, nx2 = grid["nx1"], grid["nx2"] # Nummer an Zellen in x-Richtung und y-Richtung
    dx1, dx2 = grid["dx1"], grid["dx2"] # Zellgrössen in x- und y-Richtung
    lx1, lx2 = grid["lx1"], grid["lx2"] # Lä
    
    # ====== Objekte ======
    state["no"] = 3
    Obj = state["Obj"]
    
    # Objekt 0: Festkörper / Person (adiabatisch)
    state["Obj_adiabatic"][0] = 1.0
    # ...
    
    # Objekt 1: Einlass (Überschreibungsbereich)
    state["Obj_overwrite"][1] = 1.0
    # ...
    
    # Objekt 2: Auslass (Überschreibungsbereich)
    state["Obj_overwrite"][2] = 1.0
    # ...
    
    # ====== Quellterme ======
    # - Wärmequelle
    # - CO2-Quelle


exercise_case = dict(

    lx1 = 1.0,     # L_x / L∞
    lx2 = 1.0,     # L_y / L∞
    nx1 = 100,
    nx2 = 100,
    T_max = 1.0,
    
    # Weitere Parameter
    t_end   = 200,
    nJacobi = 15,
    nPlot   = 50,

    # TODO: Passen Sie die folgenden Werte laut 1a) an
    
    # Dimensionslose Kennzahlen 
    Re = 10000,
    Pr = 1.0,
    Sc = 1.0, 
    Fr = 0.1,

    init=init_exercise
)



In [ ]:

solver_run(example_case_1) # Beispiel
# solver_run(exercise_case) # Fall aus der Aufgabenstellung